In [0]:
%sql
CREATE CATALOG IF NOT EXISTS taxi_catalog;
CREATE SCHEMA IF NOT EXISTS taxi_catalog.bronze;
CREATE VOLUME IF NOT EXISTS taxi_catalog.bronze.raw_files;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

source_path = "/Volumes/taxi_catalog/bronze/raw_files/"
print("Source Path:", source_path)

In [0]:
display(dbutils.fs.ls(source_path))

In [0]:
df_raw = (
    spark.read
    .format("parquet")
    .load(source_path)
)
display(df_raw.limit(10))

In [0]:
required_columns = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "PULocationID",
    "DOLocationID", "fare_amount", "total_amount"
]
actual_columns = set(df_raw.columns)
missing_columns = [c for c in required_columns if c not in actual_columns]

if missing_columns:
    raise ValueError(f"Schema validation failed. Missing columns: {missing_columns}")
print("Schema validation successful! All required columns present.")

In [0]:
df_bronze = (
    df_raw
    .withColumn("source_file", F.input_file_name())
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("ingestion_date", F.current_date())
)

In [0]:
df_bronze = (
    df_raw
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("ingestion_date", F.current_date())
)

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("taxi_catalog.bronze.yellow_taxi_raw")
print("Bronze table written successfully!")

In [0]:
df_check = spark.table("taxi_catalog.bronze.yellow_taxi_raw")
print("Bronze rows:", df_check.count())
display(df_check.limit(10))